In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
%run "../00.common/02.bronze-helpers.py"

In [0]:
# COMMAND ----------
 
# Load environment configuration
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# Load Bronze helper functions
 
# MAGIC %run "../00_common/02_bronze_helpers"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
source_folder = f"{s3_root_path}/landing/sprints"
 
target_path = f"{s3_root_path}/Bronze/Nisarga/sprints"
 
print("Source Folder :", source_folder)
print("Target Path   :", target_path)
 
 
# COMMAND ----------
 
# ============================================================
# DEFINE SCHEMA
# ============================================================
 
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    FloatType
)
 
sprints_schema = StructType([
    StructField("date", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])
 
 
# COMMAND ----------
 
# ============================================================
# READ ALL SPRINTS JSON FILES
# ============================================================
 
sprints_df = (
    spark.read
    .format("json")
    .schema(sprints_schema)
    .load(source_folder)
)
 
display(sprints_df)
 
 
# COMMAND ----------
 
# Check schema
 
sprints_df.printSchema()
 
print("Source Sprints Records:", sprints_df.count())
 
 
# COMMAND ----------
 
# ============================================================
# ADD INGESTION METADATA
# ============================================================
 
sprints_final_df = add_ingestion_metadata(sprints_df)
 
display(sprints_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# WRITE TO S3 BRONZE IN DELTA FORMAT
# ============================================================
 
(
    sprints_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(target_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# VALIDATE BRONZE DATA
# ============================================================
 
bronze_sprints_df = (
    spark.read
    .format("delta")
    .load(target_path)
)
 
display(bronze_sprints_df)
 
print("Total Bronze Records:", bronze_sprints_df.count())
 
print("Bronze Sprints data successfully written to:")
print(target_path)

In [0]:
%sql
 
SELECT *
FROM delta.`s3://bizf1project123/Bronze/Nisarga/sprints`;